In [2]:
"""
BBO6: CNN-Enhanced Black Box Optimisation
==========================================
Builds on BBO5 (Gradient-Enhanced SV-Informed GP with Kernel Optimisation)
and adds two CNN-based improvements targeted at F4 and F5:

Enhancement #4 - CNN Acquisition Amortisation
  A lightweight CNN pre-screens 100,000 candidates cheaply, replacing
  the expensive dim x 2 GP prediction loop. Only the top-k CNN-ranked
  candidates proceed to full GP evaluation. Gives ~10-50x speedup on
  the acquisition step with minimal quality loss.

Enhancement #5 - CNN Residual GP Correction
  A residual CNN learns the GP's leave-one-out prediction errors and
  corrects both mean and uncertainty before acquisition scoring:
      final_prediction(x) = GP_mean(x) + CNN_residual(x)
      final_std(x)        = GP_std(x)  * CNN_uncertainty_scale(x)

Why F4 and F5?
  F4: UCB = -8.49 vs current best = +0.14  (GP pointing away from optimum)
  F5: UCB = 3014  vs current best = 4413   (GP can't locate boundary near optimum)
  Both are high-curvature landscapes where Matern kernels lose calibration.
"""

import numpy as np
import warnings
warnings.filterwarnings('ignore')

# GP stack
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, ConstantKernel, WhiteKernel
from sklearn.model_selection import cross_val_score
from scipy.optimize import minimize

# CNN stack
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

print(f"PyTorch version: {torch.__version__}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")


# ══════════════════════════════════════════════════════════════════════════════
# PART 1: GP INFRASTRUCTURE  (unchanged from BBO5)
# ══════════════════════════════════════════════════════════════════════════════

class GradientEnhancedGP:
    """
    Gradient-Enhanced Gaussian Process.
    Augments training data with virtual gradient observations:
        f(x + ε·eᵢ) ≈ f(x) + ε·gᵢ
    Turns N observations into N + 2·N·D effective training points.
    """

    def __init__(self, kernel, alpha=1e-6, normalize_y=True):
        self.kernel     = kernel
        self.alpha      = alpha
        self.normalize_y = normalize_y
        self.gp         = None
        self.X_train    = None
        self.y_train    = None
        self.y_mean     = 0.0
        self.y_std      = 1.0

    def _augment_training_data(self, X, y, gradients, epsilon=0.01):
        n, dim = X.shape
        X_aug = [X]
        y_aug = [y]
        for d in range(dim):
            X_plus       = X.copy(); X_plus[:, d]  += epsilon
            X_minus      = X.copy(); X_minus[:, d] -= epsilon
            y_aug.append(y + epsilon * gradients[:, d])
            y_aug.append(y - epsilon * gradients[:, d])
            X_aug.extend([X_plus, X_minus])
        return np.vstack(X_aug), np.concatenate(y_aug)

    def fit(self, X, y, gradients=None):
        self.X_train = X
        self.y_train = y

        if self.normalize_y:
            self.y_mean = y.mean()
            self.y_std  = y.std() if y.std() > 0 else 1.0
            y_norm      = (y - self.y_mean) / self.y_std
        else:
            y_norm = y

        if gradients is not None:
            grad_norm = gradients / self.y_std if self.normalize_y else gradients
            X_aug, y_aug = self._augment_training_data(X, y_norm, grad_norm)
            print(f"  Gradient-enhanced GP: {len(X)} obs → {len(X_aug)} effective points")
        else:
            X_aug, y_aug = X, y_norm

        self.gp = GaussianProcessRegressor(
            kernel=self.kernel, alpha=self.alpha,
            n_restarts_optimizer=10, normalize_y=False
        )
        self.gp.fit(X_aug, y_aug)
        return self

    def predict(self, X, return_std=False):
        if return_std:
            mu, std = self.gp.predict(X, return_std=True)
            if self.normalize_y:
                mu  = mu  * self.y_std + self.y_mean
                std = std * self.y_std
            return mu, std
        else:
            mu = self.gp.predict(X)
            if self.normalize_y:
                mu = mu * self.y_std + self.y_mean
            return mu

    @property
    def kernel_(self):
        return self.gp.kernel_ if self.gp else self.kernel

    def log_marginal_likelihood(self):
        return self.gp.log_marginal_likelihood() if self.gp else -np.inf


def estimate_gradients_numerically(gp, X_obs, epsilon=0.01, bounds=None):
    """Finite-difference gradient estimation from GP predictions."""
    n, dim = X_obs.shape
    gradients = np.zeros((n, dim))
    lows  = np.array([b[0] for b in bounds]) if bounds else None
    highs = np.array([b[1] for b in bounds]) if bounds else None

    for d in range(dim):
        X_plus  = X_obs.copy()
        X_minus = X_obs.copy()
        if lows is not None:
            X_plus[:, d]  = np.clip(X_plus[:, d]  + epsilon, lows[d], highs[d])
            X_minus[:, d] = np.clip(X_minus[:, d] - epsilon, lows[d], highs[d])
        else:
            X_plus[:, d]  += epsilon
            X_minus[:, d] -= epsilon
        gradients[:, d] = (gp.predict(X_plus) - gp.predict(X_minus)) / (2 * epsilon)

    return gradients


def select_best_kernel(X_obs, y_obs, dim):
    """Cross-validation kernel selection: RBF, Matern-2.5, Matern-1.5 (all ARD)."""
    kernels = {
        'RBF_ARD': (
            ConstantKernel(1.0, (0.1, 10.0)) *
            RBF(np.ones(dim), (0.01, 20.0)) +
            WhiteKernel(1e-5, (1e-10, 1e-2))
        ),
        'Matern_2.5_ARD': (
            ConstantKernel(1.0, (0.1, 10.0)) *
            Matern(np.ones(dim), (0.01, 20.0), nu=2.5) +
            WhiteKernel(1e-5, (1e-10, 1e-2))
        ),
        'Matern_1.5_ARD': (
            ConstantKernel(1.0, (0.1, 10.0)) *
            Matern(np.ones(dim), (0.01, 20.0), nu=1.5) +
            WhiteKernel(1e-5, (1e-10, 1e-2))
        ),
    }

    best_name, best_kernel, best_gp, best_score = None, None, None, -np.inf

    for name, kernel in kernels.items():
        try:
            gp_test = GaussianProcessRegressor(
                kernel=kernel, alpha=1e-6,
                n_restarts_optimizer=20, normalize_y=True
            )
            if len(X_obs) >= 5:
                scores = cross_val_score(
                    gp_test, X_obs, y_obs,
                    cv=min(5, len(X_obs)),
                    scoring='neg_mean_squared_error'
                )
                score = scores.mean()
            else:
                gp_test.fit(X_obs, y_obs)
                score = gp_test.score(X_obs, y_obs)

            gp_test.fit(X_obs, y_obs)
            print(f"  {name}: CV={score:.4f}  kernel={gp_test.kernel_}")

            if score > best_score:
                best_score, best_name = score, name
                best_kernel, best_gp  = gp_test.kernel_, gp_test
                print("    ✓ New best")
        except Exception as e:
            print(f"  {name}: FAILED — {e}")

    return best_name, best_kernel, best_gp


# ══════════════════════════════════════════════════════════════════════════════
# PART 2: CNN ARCHITECTURES
# ══════════════════════════════════════════════════════════════════════════════

class GradientPredictorCNN(nn.Module):
    """
    Enhancement #4 — CNN Acquisition Amortiser

    Maps a candidate point x ∈ [0,1]^D to a predicted gradient magnitude,
    trained on (X_obs, grad_magnitudes) pairs from the current GP.

    Architecture: two 1-D convolutional layers over the feature dimension,
    followed by fully-connected layers. The convolutions learn local
    correlations between adjacent feature dimensions (useful when features
    are ordered or partially correlated).

    Input:  (batch, D)
    Output: (batch, 1)  — predicted gradient magnitude score ∈ ℝ+
    """

    def __init__(self, dim, hidden=64, dropout=0.2):
        super().__init__()
        self.dim = dim

        # Treat D features as a 1-D sequence of length D with 1 channel
        self.conv_block = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=min(3, dim), padding=min(1, dim // 2)),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=min(3, dim), padding=min(1, dim // 2)),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),          # → (batch, 64, 1)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),                     # → (batch, 64)
            nn.Linear(64, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, 1),
            nn.Softplus(),                    # ensures output ≥ 0
        )

    def forward(self, x):
        # x: (batch, D)  →  (batch, 1, D)  for Conv1d
        x = x.unsqueeze(1)
        x = self.conv_block(x)
        return self.fc(x)


class ResidualCorrectorCNN(nn.Module):
    """
    Enhancement #5 — CNN Residual GP Corrector

    Learns the GP's systematic prediction error using leave-one-out (LOO)
    residuals, then corrects GP mean and uncertainty at candidate points.

    Two output heads:
      - residual_head:  additive correction to GP mean
      - scale_head:     multiplicative correction to GP std (> 0)

    Architecture: shared MLP backbone with convolutional feature mixing,
    then two separate heads.

    Input:  (batch, D) — normalised to [0,1]
    Output: (batch, 2) — [residual, uncertainty_scale]
    """

    def __init__(self, dim, hidden=64, dropout=0.2):
        super().__init__()
        self.dim = dim

        self.backbone = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
        )

        self.residual_head = nn.Sequential(
            nn.Linear(hidden // 2, 16),
            nn.ReLU(),
            nn.Linear(16, 1),               # additive correction ∈ ℝ
        )

        self.scale_head = nn.Sequential(
            nn.Linear(hidden // 2, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Softplus(),                  # multiplicative scale > 0
        )

    def forward(self, x):
        h        = self.backbone(x)
        residual = self.residual_head(h)
        scale    = self.scale_head(h) + 0.5  # bias toward scale ≈ 1
        return torch.cat([residual, scale], dim=1)


# ══════════════════════════════════════════════════════════════════════════════
# PART 3: CNN TRAINING UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def train_gradient_cnn(X_obs, grad_magnitudes, dim, epochs=200, lr=1e-3, verbose=True):
    """
    Train the GradientPredictorCNN on observed (X, gradient_magnitude) pairs.

    Uses bootstrap augmentation to expand the small training set:
    each observation is jittered with Gaussian noise to create 20x more
    synthetic training points, preventing overfitting on small datasets.
    """
    # Normalise targets to [0,1] for stable training
    g_min, g_max = grad_magnitudes.min(), grad_magnitudes.max()
    g_range = g_max - g_min if g_max > g_min else 1.0
    grad_norm = (grad_magnitudes - g_min) / g_range

    # Bootstrap augmentation: jitter observations
    n_aug  = 20
    noise  = 0.02
    X_aug  = np.vstack([X_obs + np.random.randn(*X_obs.shape) * noise
                        for _ in range(n_aug)])
    X_aug  = np.clip(X_aug, 0, 1)
    g_aug  = np.tile(grad_norm, n_aug)

    X_t = torch.FloatTensor(X_aug).to(DEVICE)
    g_t = torch.FloatTensor(g_aug).unsqueeze(1).to(DEVICE)

    model     = GradientPredictorCNN(dim).to(DEVICE)
    optimiser = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=epochs)
    criterion = nn.MSELoss()

    dataset = TensorDataset(X_t, g_t)
    loader  = DataLoader(dataset, batch_size=min(64, len(X_aug)), shuffle=True)

    model.train()
    losses = []
    for epoch in range(epochs):
        epoch_loss = 0.0
        for xb, gb in loader:
            optimiser.zero_grad()
            pred = model(xb)
            loss = criterion(pred, gb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimiser.step()
            epoch_loss += loss.item()
        scheduler.step()
        losses.append(epoch_loss / len(loader))

    if verbose:
        print(f"  GradCNN trained: final loss = {losses[-1]:.6f}")

    # Store normalisation params for inverse transform at inference time
    model.g_min   = g_min
    model.g_range = g_range
    model.eval()
    return model, losses


def compute_loo_residuals(gp_base, X_obs, y_obs):
    """
    Leave-one-out residuals for training the ResidualCorrectorCNN.
    For each point i, refit GP on all points except i, predict at i,
    and record (y_true - y_pred) and the LOO std.

    Returns arrays of shape (N,) for residuals and loo_std.
    """
    n   = len(y_obs)
    dim = X_obs.shape[1]
    residuals = np.zeros(n)
    loo_stds  = np.zeros(n)

    for i in range(n):
        mask      = np.ones(n, dtype=bool); mask[i] = False
        X_train_i = X_obs[mask]
        y_train_i = y_obs[mask]

        try:
            gp_loo = GaussianProcessRegressor(
                kernel=gp_base.kernel_, alpha=1e-6,
                n_restarts_optimizer=3, normalize_y=True
            )
            gp_loo.fit(X_train_i, y_train_i)
            mu_i, std_i   = gp_loo.predict(X_obs[i:i+1], return_std=True)
            residuals[i]  = y_obs[i] - mu_i[0]
            loo_stds[i]   = std_i[0]
        except Exception:
            residuals[i] = 0.0
            loo_stds[i]  = 1.0

    return residuals, loo_stds


def train_residual_cnn(X_obs, residuals, loo_stds, y_obs, dim,
                       epochs=300, lr=1e-3, verbose=True):
    """
    Train the ResidualCorrectorCNN on LOO residuals.

    Targets:
      head 1: normalised residual (additive GP mean correction)
      head 2: |residual| / loo_std  (uncertainty scale factor)
    """
    y_std = y_obs.std() if y_obs.std() > 0 else 1.0

    # Normalise residuals by output scale
    res_norm  = residuals / y_std
    # Uncertainty scale: how much did the GP under/over-estimate spread?
    unc_scale = np.abs(residuals) / (loo_stds + 1e-8)
    unc_scale = np.clip(unc_scale, 0.1, 5.0)   # cap extremes

    # Bootstrap augmentation
    n_aug = 30
    noise = 0.02
    X_aug = np.vstack([X_obs + np.random.randn(*X_obs.shape) * noise
                       for _ in range(n_aug)])
    X_aug     = np.clip(X_aug, 0, 1)
    res_aug   = np.tile(res_norm, n_aug)
    scale_aug = np.tile(unc_scale, n_aug)

    X_t   = torch.FloatTensor(X_aug).to(DEVICE)
    tgt_t = torch.FloatTensor(
        np.stack([res_aug, scale_aug], axis=1)
    ).to(DEVICE)

    model     = ResidualCorrectorCNN(dim).to(DEVICE)
    optimiser = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=epochs)

    dataset = TensorDataset(X_t, tgt_t)
    loader  = DataLoader(dataset, batch_size=min(64, len(X_aug)), shuffle=True)

    model.train()
    losses = []
    for epoch in range(epochs):
        epoch_loss = 0.0
        for xb, tb in loader:
            optimiser.zero_grad()
            pred = model(xb)
            # Loss = MSE on residual + MSE on uncertainty scale
            loss = nn.MSELoss()(pred[:, 0:1], tb[:, 0:1]) + \
                   0.5 * nn.MSELoss()(pred[:, 1:2], tb[:, 1:2])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimiser.step()
            epoch_loss += loss.item()
        scheduler.step()
        losses.append(epoch_loss / len(loader))

    if verbose:
        print(f"  ResidualCNN trained: final loss = {losses[-1]:.6f}")

    model.y_std = y_std
    model.eval()
    return model, losses


# ══════════════════════════════════════════════════════════════════════════════
# PART 4: CNN INFERENCE HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def cnn_predict_gradients(grad_cnn, X_candidates, batch_size=4096):
    """
    Run GradientPredictorCNN over all candidates in batches.
    Returns predicted gradient magnitudes in original scale.
    """
    grad_cnn.eval()
    preds = []
    with torch.no_grad():
        for start in range(0, len(X_candidates), batch_size):
            xb   = torch.FloatTensor(
                X_candidates[start:start + batch_size]
            ).to(DEVICE)
            pred = grad_cnn(xb).cpu().numpy().ravel()
            preds.append(pred)
    scores = np.concatenate(preds)
    # Inverse-normalise
    return scores * grad_cnn.g_range + grad_cnn.g_min


def cnn_correct_gp(residual_cnn, gp, X_candidates, batch_size=4096):
    """
    Apply ResidualCorrectorCNN corrections to GP mean and std predictions.

    Returns corrected (mu_corrected, std_corrected) arrays.
    """
    residual_cnn.eval()
    corrections = []
    with torch.no_grad():
        for start in range(0, len(X_candidates), batch_size):
            xb   = torch.FloatTensor(
                X_candidates[start:start + batch_size]
            ).to(DEVICE)
            pred = residual_cnn(xb).cpu().numpy()
            corrections.append(pred)
    corrections = np.vstack(corrections)

    # Get base GP predictions
    mu_gp, std_gp = gp.predict(X_candidates, return_std=True)

    # Apply corrections
    residual      = corrections[:, 0] * residual_cnn.y_std
    unc_scale     = corrections[:, 1]

    mu_corrected  = mu_gp + residual
    std_corrected = std_gp * np.clip(unc_scale, 0.1, 5.0)

    return mu_corrected, std_corrected


# ══════════════════════════════════════════════════════════════════════════════
# PART 5: MAIN BBO PIPELINE  (CNN-enhanced)
# ══════════════════════════════════════════════════════════════════════════════

def normalize(x):
    r = x.max() - x.min()
    return (x - x.min()) / r if r > 0 else np.zeros_like(x)


def cnn_enhanced_bbo(
    X_obs, y_obs, bounds,
    kappa=4.0,
    sv_weight=0.4,
    n_random_samples=100_000,
    cnn_prescreen_k=5000,      # candidates passed from CNN to full GP eval
    use_gradient_enhancement=True,
    use_cnn_amortisation=True,   # Enhancement #4
    use_residual_correction=True, # Enhancement #5
    fn_label="F?",
    cnn_epochs=250,
):
    """
    Full pipeline:
      1. Kernel optimisation (BBO5)
      2. Gradient enhancement (BBO5)
      3. CNN Residual Corrector training (Enhancement #5)
      4. SV-like point identification (BBO5)
      5. CNN Acquisition Amortisation — pre-screen 100k candidates (Enhancement #4)
      6. Corrected UCB scoring on top-k candidates (Enhancement #5)
      7. scipy L-BFGS-B local refinement (BBO5)
    """
    dim  = len(bounds)
    lows  = np.array([b[0] for b in bounds])
    highs = np.array([b[1] for b in bounds])
    eps   = 0.01

    print("=" * 70)
    print(f"CNN-ENHANCED GP BBO  —  {fn_label}  ({dim}D, {len(y_obs)} obs)")
    print("=" * 70)

    # ── STEP 1: Kernel Optimisation ──────────────────────────────────────────
    print("\n── STEP 1: KERNEL OPTIMISATION")
    best_name, best_kernel, best_gp_base = select_best_kernel(X_obs, y_obs, dim)
    print(f"\n  Best kernel: {best_name}")

    # ── STEP 2: Gradient Enhancement ────────────────────────────────────────
    print("\n── STEP 2: GRADIENT ENHANCEMENT")
    if use_gradient_enhancement:
        gradients_est = estimate_gradients_numerically(
            best_gp_base, X_obs, epsilon=eps, bounds=bounds
        )
        grad_mags_obs = np.linalg.norm(gradients_est, axis=1)
        print(f"  Gradient magnitudes — mean={grad_mags_obs.mean():.4f}  "
              f"min={grad_mags_obs.min():.4f}  max={grad_mags_obs.max():.4f}")

        gp_enhanced = GradientEnhancedGP(
            kernel=best_kernel, alpha=1e-6, normalize_y=True
        )
        gp_enhanced.fit(X_obs, y_obs, gradients=gradients_est)
        gp = gp_enhanced
    else:
        gradients_est  = None
        grad_mags_obs  = np.zeros(len(y_obs))
        gp             = best_gp_base

    # ── STEP 3: Train CNN Residual Corrector (Enhancement #5) ───────────────
    residual_cnn = None
    if use_residual_correction and len(X_obs) >= 5:
        print("\n── STEP 3: CNN RESIDUAL CORRECTOR TRAINING  [Enhancement #5]")
        print("  Computing LOO residuals from base GP...")
        residuals, loo_stds = compute_loo_residuals(best_gp_base, X_obs, y_obs)
        print(f"  LOO residuals — mean={residuals.mean():.4f}  "
              f"std={residuals.std():.4f}  max_abs={np.abs(residuals).max():.4f}")

        residual_cnn, res_losses = train_residual_cnn(
            X_obs, residuals, loo_stds, y_obs, dim,
            epochs=cnn_epochs, verbose=True
        )
    else:
        print("\n── STEP 3: Residual correction skipped (need ≥5 obs)")

    # ── STEP 4: Train Gradient Predictor CNN (Enhancement #4) ───────────────
    grad_cnn = None
    if use_cnn_amortisation and gradients_est is not None:
        print("\n── STEP 4: CNN GRADIENT PREDICTOR TRAINING  [Enhancement #4]")
        grad_cnn, grad_losses = train_gradient_cnn(
            X_obs, grad_mags_obs, dim,
            epochs=cnn_epochs, verbose=True
        )

    # ── STEP 5: SV-like Point Identification ────────────────────────────────
    print("\n── STEP 5: SV-LIKE POINT ANALYSIS")
    sv_scores     = normalize(grad_mags_obs) if gradients_est is not None \
                    else np.zeros(len(y_obs))
    top_sv_idx    = np.argsort(sv_scores)[::-1][:min(5, len(y_obs))]
    print(f"  Top {len(top_sv_idx)} SV-like points:")
    for rank, idx in enumerate(top_sv_idx):
        print(f"    Rank {rank+1}: P{idx}  val={y_obs[idx]:.4f}  "
              f"grad={grad_mags_obs[idx]:.4f}  sv={sv_scores[idx]:.4f}")

    # ── STEP 6: CNN Pre-Screening then Corrected UCB Scoring ─────────────────
    print(f"\n── STEP 6: ACQUISITION  "
          f"(CNN pre-screen {n_random_samples:,} → {cnn_prescreen_k:,} → full GP)")

    X_candidates = np.random.uniform(
        low=lows, high=highs, size=(n_random_samples, dim)
    )

    # ── 6a: CNN pre-screen (Enhancement #4) ──────────────────────────────────
    if grad_cnn is not None:
        print("  CNN pre-screening candidates...")
        cnn_grad_scores = cnn_predict_gradients(grad_cnn, X_candidates)
        cnn_ucb_quick   = normalize(cnn_grad_scores)
        # Select top-k by CNN score for full GP evaluation
        top_k_idx       = np.argsort(cnn_ucb_quick)[::-1][:cnn_prescreen_k]
        X_shortlisted   = X_candidates[top_k_idx]
        print(f"  CNN shortlisted {cnn_prescreen_k:,} candidates "
              f"(saved {n_random_samples - cnn_prescreen_k:,} GP evaluations)")
    else:
        X_shortlisted = X_candidates
        cnn_grad_scores = np.zeros(n_random_samples)
        top_k_idx = np.arange(n_random_samples)
        print("  CNN amortisation skipped — using full candidate set")

    # ── 6b: GP predictions on shortlisted candidates ─────────────────────────
    print(f"  Running GP on {len(X_shortlisted):,} shortlisted candidates...")

    if residual_cnn is not None:
        # Enhancement #5: corrected predictions
        mu_cand, std_cand = cnn_correct_gp(
            residual_cnn, gp, X_shortlisted
        )
        print("  Applied CNN residual correction to GP predictions")
    else:
        mu_cand, std_cand = gp.predict(X_shortlisted, return_std=True)

    # ── 6c: Gradient scores on shortlisted candidates ────────────────────────
    if grad_cnn is not None:
        grad_shortlisted = cnn_grad_scores[top_k_idx]
    else:
        # Fallback: compute numerically on shortlisted set only
        grad_shortlisted = np.zeros(len(X_shortlisted))
        batch_size = 1000
        for start in range(0, len(X_shortlisted), batch_size):
            X_b = X_shortlisted[start:start + batch_size]
            g_b = np.zeros(len(X_b))
            for d in range(dim):
                X_p = X_b.copy(); X_p[:, d] = np.clip(X_p[:, d] + eps, lows[d], highs[d])
                X_m = X_b.copy(); X_m[:, d] = np.clip(X_m[:, d] - eps, lows[d], highs[d])
                g_b += ((gp.predict(X_p) - gp.predict(X_m)) / (2 * eps)) ** 2
            grad_shortlisted[start:start + batch_size] = np.sqrt(g_b)

    # ── 6d: Combined SV-UCB acquisition score ────────────────────────────────
    ucb_scores    = mu_cand + kappa * std_cand
    ucb_norm      = normalize(ucb_scores)
    grad_norm_sl  = normalize(grad_shortlisted)
    sv_ucb        = (1 - sv_weight) * ucb_norm + sv_weight * grad_norm_sl

    best_idx   = np.argmax(sv_ucb)
    next_point = X_shortlisted[best_idx].copy()

    # ── STEP 7: scipy L-BFGS-B Local Refinement ─────────────────────────────
    print("\n── STEP 7: LOCAL REFINEMENT (L-BFGS-B)")

    def negative_sv_ucb(x):
        x2d    = np.atleast_2d(x)
        if residual_cnn is not None:
            mu, std = cnn_correct_gp(residual_cnn, gp, x2d)
        else:
            mu, std = gp.predict(x2d, return_std=True)
        grad_x = 0.0
        for d in range(dim):
            xp = x2d.copy(); xp[0, d] = min(xp[0, d] + eps, highs[d])
            xm = x2d.copy(); xm[0, d] = max(xm[0, d] - eps, lows[d])
            if residual_cnn is not None:
                mp, _ = cnn_correct_gp(residual_cnn, gp, xp)
                mm, _ = cnn_correct_gp(residual_cnn, gp, xm)
            else:
                mp = gp.predict(xp); mm = gp.predict(xm)
            grad_x += ((mp[0] - mm[0]) / (2 * eps)) ** 2
        ucb_val = mu[0] + kappa * std[0]
        sv_val  = (1 - sv_weight) * ucb_val + sv_weight * np.sqrt(grad_x)
        return -sv_val

    result = minimize(
        negative_sv_ucb, x0=next_point,
        method='L-BFGS-B', bounds=bounds
    )
    if result.success and -result.fun > sv_ucb[best_idx]:
        next_point = result.x
        print("  ✓ Refinement improved the acquisition score")
    else:
        print("  ○ Random search candidate retained")

    # ── Final predictions ─────────────────────────────────────────────────────
    if residual_cnn is not None:
        mu_f, std_f = cnn_correct_gp(
            residual_cnn, gp, next_point.reshape(1, -1)
        )
    else:
        mu_f, std_f = gp.predict(next_point.reshape(1, -1), return_std=True)
    mu_f, std_f = float(mu_f[0]), float(std_f[0])

    # ── Summary ───────────────────────────────────────────────────────────────
    best_obs_idx = np.argmax(y_obs)
    print(f"\n{'=' * 70}")
    print(f"RESULTS — {fn_label}")
    print(f"{'=' * 70}")
    print(f"  Kernel:              {best_name}")
    print(f"  Gradient enhancement: {'YES' if use_gradient_enhancement else 'NO'}")
    print(f"  CNN amortisation:    {'YES' if grad_cnn is not None else 'NO'}")
    print(f"  CNN residual corr:   {'YES' if residual_cnn is not None else 'NO'}")
    print(f"\n  Next point to sample:")
    for d, v in enumerate(next_point):
        print(f"    x{d}: {v:.6f}")
    print(f"\n  Expected performance:")
    print(f"    Mean:         {mu_f:.6f}")
    print(f"    Uncertainty:  {std_f:.6f}")
    print(f"    UCB:          {mu_f + kappa * std_f:.6f}")
    print(f"\n  Current best:  {y_obs[best_obs_idx]:.6f}  at {X_obs[best_obs_idx]}")
    improvement = (mu_f + kappa * std_f) - y_obs[best_obs_idx]
    print(f"  Potential improvement: {improvement:+.6f}")

    return {
        'fn_label':             fn_label,
        'next_point':           next_point,
        'mean':                 mu_f,
        'std':                  std_f,
        'ucb':                  mu_f + kappa * std_f,
        'kernel':               best_name,
        'gp':                   gp,
        'grad_cnn':             grad_cnn,
        'residual_cnn':         residual_cnn,
        'sv_scores':            sv_scores,
        'top_sv_indices':       top_sv_idx,
        'gradients':            gradients_est,
        'improvement':          improvement,
    }


# ══════════════════════════════════════════════════════════════════════════════
# PART 6: DIAGNOSTICS & VISUALISATION
# ══════════════════════════════════════════════════════════════════════════════

def plot_cnn_diagnostics(results, X_obs, y_obs, fn_label, bounds, kappa=4.0, sv_weight=0.4):
    """
    Two-row diagnostic plot for a single function.

    ROW 1 — Model accuracy (what the GP knows):
      Panel 1: GP mean (raw vs corrected) along most informative dimension
      Panel 2: CNN gradient predictor accuracy vs numerical gradient
      Panel 3: Residual correction magnitude per training point

    ROW 2 — Exploration vs Exploitation:
      Panel 4: GP mean μ(x)  — exploitation signal along best dimension
      Panel 5: GP uncertainty σ(x) — exploration signal, with next point marked
      Panel 6: UCB acquisition = μ + κσ decomposed into exploit vs explore share,
               with next point and current best marked
    """
    dim          = X_obs.shape[1]
    gp           = results['gp']
    grad_cnn     = results['grad_cnn']
    residual_cnn = results['residual_cnn']
    gradients    = results['gradients']
    next_point   = results['next_point']

    # ── Shared sweep along most informative dimension ─────────────────────────
    if gradients is not None:
        best_dim = np.argmax(np.std(gradients, axis=0))
    else:
        best_dim = 0

    n_sweep = 300
    sweep   = np.tile(X_obs.mean(axis=0), (n_sweep, 1))
    x_axis  = np.linspace(bounds[best_dim][0], bounds[best_dim][1], n_sweep)
    sweep[:, best_dim] = x_axis

    # Raw GP predictions along sweep
    mu_raw, std_raw = gp.predict(sweep, return_std=True)

    # Corrected predictions (if available)
    if residual_cnn is not None:
        mu_use, std_use = cnn_correct_gp(residual_cnn, gp, sweep)
    else:
        mu_use, std_use = mu_raw.copy(), std_raw.copy()

    ucb_exploit = mu_use                    # exploitation component
    ucb_explore = kappa * std_use           # exploration component
    ucb_total   = ucb_exploit + ucb_explore

    # Next point projected onto best_dim for vertical marker
    next_x = next_point[best_dim]
    best_obs_idx = np.argmax(y_obs)
    best_x = X_obs[best_obs_idx, best_dim]

    # ── Figure layout: 2 rows × 3 columns ────────────────────────────────────
    fig = plt.figure(figsize=(18, 9))
    fig.suptitle(
        f"{fn_label}  —  CNN Diagnostics & Exploration vs Exploitation  "
        f"(dim {best_dim} sweep, kappa={kappa})",
        fontsize=13, fontweight='bold', y=1.01
    )
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

    # ════════════════════════════════════════════════════════
    # ROW 1: Model Accuracy
    # ════════════════════════════════════════════════════════

    # ── Panel 1: GP mean raw vs corrected ────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.fill_between(x_axis, mu_raw - std_raw, mu_raw + std_raw,
                     alpha=0.15, color='steelblue')
    ax1.plot(x_axis, mu_raw, color='steelblue', linewidth=2, label='GP mean (raw)')

    if residual_cnn is not None:
        ax1.fill_between(x_axis, mu_use - std_use, mu_use + std_use,
                         alpha=0.15, color='tomato')
        ax1.plot(x_axis, mu_use, color='tomato', linewidth=2,
                 linestyle='--', label='GP mean (corrected)')

    ax1.scatter(X_obs[:, best_dim], y_obs, c='black', zorder=5, s=25,
                label='Observations')
    ax1.axvline(next_x, color='green', linewidth=1.5, linestyle=':',
                label=f'Next point (x{best_dim}={next_x:.3f})')
    ax1.axvline(best_x, color='gold', linewidth=1.5, linestyle='-.',
                label=f'Current best (x{best_dim}={best_x:.3f})')
    ax1.set_xlabel(f'x{best_dim}')
    ax1.set_ylabel('f(x)')
    ax1.set_title('Panel 1 — GP Mean\n(raw vs corrected)')
    ax1.legend(fontsize=6.5)

    # ── Panel 2: CNN gradient predictor accuracy ──────────────────────────────
    ax2 = fig.add_subplot(gs[0, 1])
    if grad_cnn is not None and gradients is not None:
        numerical_grads = np.linalg.norm(gradients, axis=1)
        cnn_grads       = cnn_predict_gradients(grad_cnn, X_obs)
        sc = ax2.scatter(numerical_grads, cnn_grads, c=y_obs, cmap='viridis',
                         s=60, edgecolors='k', linewidths=0.5, zorder=5)
        lim = max(numerical_grads.max(), cnn_grads.max()) * 1.05
        ax2.plot([0, lim], [0, lim], 'r--', linewidth=1, label='Perfect fit')
        ax2.set_xlabel('Numerical gradient magnitude')
        ax2.set_ylabel('CNN predicted gradient')
        corr = np.corrcoef(numerical_grads, cnn_grads)[0, 1]
        ax2.text(0.05, 0.93, f'Pearson r = {corr:.3f}',
                 transform=ax2.transAxes, fontsize=9,
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
        ax2.legend(fontsize=8)
        plt.colorbar(sc, ax=ax2, label='f(x)', pad=0.02)
    else:
        ax2.text(0.5, 0.5, 'CNN amortisation\nnot active',
                 ha='center', va='center', transform=ax2.transAxes, fontsize=11)
    ax2.set_title('Panel 2 — Enhancement #4\nGrad CNN vs Numerical')

    # ── Panel 3: Residual corrections at training points ─────────────────────
    ax3 = fig.add_subplot(gs[0, 2])
    if residual_cnn is not None:
        mu_raw_obs, _ = gp.predict(X_obs, return_std=True)
        mu_corr_obs, _ = cnn_correct_gp(residual_cnn, gp, X_obs)
        corrections = mu_corr_obs - mu_raw_obs
        colors = ['tomato' if c < 0 else 'steelblue' for c in corrections]
        ax3.bar(range(len(corrections)), corrections, color=colors,
                edgecolor='k', linewidth=0.4, alpha=0.85)
        ax3.axhline(0, color='black', linewidth=1)
        ax3.set_xlabel('Observation index')
        ax3.set_ylabel('CNN correction (corrected − raw)')
        ax3.text(0.05, 0.95, f'Mean |corr|: {np.abs(corrections).mean():.4f}',
                 transform=ax3.transAxes, fontsize=8, va='top',
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
    else:
        ax3.text(0.5, 0.5, 'Residual correction\nnot active',
                 ha='center', va='center', transform=ax3.transAxes, fontsize=11)
    ax3.set_title('Panel 3 — Enhancement #5\nResidual Corrections at Obs Points')

    # ════════════════════════════════════════════════════════
    # ROW 2: Exploration vs Exploitation
    # ════════════════════════════════════════════════════════

    # ── Panel 4: Exploitation signal — GP mean μ(x) ──────────────────────────
    ax4 = fig.add_subplot(gs[1, 0])
    ax4.fill_between(x_axis, mu_use - std_use, mu_use + std_use,
                     alpha=0.2, color='steelblue', label='±1σ uncertainty')
    ax4.plot(x_axis, mu_use, color='steelblue', linewidth=2.5, label='μ(x) — exploit signal')
    ax4.scatter(X_obs[:, best_dim], y_obs, c='black', zorder=5, s=25,
                alpha=0.6, label='Observations')
    ax4.axvline(next_x, color='green', linewidth=2, linestyle=':',
                label=f'Next point')
    ax4.axvline(best_x, color='gold', linewidth=2, linestyle='-.',
                label=f'Current best')

    # Shade the exploitation zone (top 20% of mean values)
    exploit_thresh = np.percentile(mu_use, 80)
    ax4.fill_between(x_axis, exploit_thresh, mu_use,
                     where=(mu_use >= exploit_thresh),
                     alpha=0.25, color='green', label='Top 20% exploit zone')
    ax4.axhline(exploit_thresh, color='green', linewidth=0.8,
                linestyle='--', alpha=0.6)

    ax4.set_xlabel(f'x{best_dim}')
    ax4.set_ylabel('μ(x) — predicted mean')
    ax4.set_title('Panel 4 — EXPLOITATION\nGP Mean μ(x): where is f(x) high?')
    ax4.legend(fontsize=6.5)

    # Annotate exploit vs explore balance at next point
    next_sweep = np.tile(X_obs.mean(axis=0), (1, 1))
    next_sweep[0, best_dim] = next_x
    if residual_cnn is not None:
        mu_next, std_next = cnn_correct_gp(residual_cnn, gp, next_sweep)
    else:
        mu_next, std_next = gp.predict(next_sweep, return_std=True)
    mu_next, std_next = float(mu_next[0]), float(std_next[0])

    ax4.annotate(
        f'μ={mu_next:.3f}',
        xy=(next_x, mu_next),
        xytext=(next_x + 0.05, mu_next + (mu_use.max() - mu_use.min()) * 0.08),
        fontsize=8, color='green',
        arrowprops=dict(arrowstyle='->', color='green', lw=1.2)
    )

    # ── Panel 5: Exploration signal — GP uncertainty σ(x) ────────────────────
    ax5 = fig.add_subplot(gs[1, 1])
    ax5.plot(x_axis, std_use, color='darkorange', linewidth=2.5,
             label='σ(x) — explore signal')
    ax5.fill_between(x_axis, 0, std_use, alpha=0.2, color='darkorange')

    # Shade the exploration zone (top 20% of std values)
    explore_thresh = np.percentile(std_use, 80)
    ax5.fill_between(x_axis, explore_thresh, std_use,
                     where=(std_use >= explore_thresh),
                     alpha=0.35, color='darkorange', label='Top 20% explore zone')
    ax5.axhline(explore_thresh, color='darkorange', linewidth=0.8,
                linestyle='--', alpha=0.6)

    # Mark observation locations as low-uncertainty anchors
    for xo in X_obs[:, best_dim]:
        ax5.axvline(xo, color='grey', linewidth=0.4, alpha=0.4)
    ax5.axvline(X_obs[0, best_dim], color='grey', linewidth=0.4,
                alpha=0.4, label='Observation locations (↓σ)')

    ax5.axvline(next_x, color='green', linewidth=2, linestyle=':',
                label=f'Next point (σ={std_next:.4f})')
    ax5.axvline(best_x, color='gold', linewidth=2, linestyle='-.',
                label='Current best')

    ax5.annotate(
        f'σ={std_next:.4f}',
        xy=(next_x, std_next),
        xytext=(next_x + 0.05, std_next + std_use.max() * 0.1),
        fontsize=8, color='green',
        arrowprops=dict(arrowstyle='->', color='green', lw=1.2)
    )

    ax5.set_xlabel(f'x{best_dim}')
    ax5.set_ylabel('σ(x) — uncertainty')
    ax5.set_title('Panel 5 — EXPLORATION\nGP Uncertainty σ(x): where is f(x) unknown?')
    ax5.legend(fontsize=6.5)

    # ── Panel 6: UCB = μ + κσ decomposed ─────────────────────────────────────
    ax6 = fig.add_subplot(gs[1, 2])

    # Stacked area: exploitation (μ) on bottom, exploration (κσ) on top
    ax6.fill_between(x_axis, 0, ucb_exploit,
                     alpha=0.4, color='steelblue', label=f'Exploit: μ(x)')
    ax6.fill_between(x_axis, ucb_exploit, ucb_total,
                     alpha=0.4, color='darkorange', label=f'Explore: κ·σ(x)  (κ={kappa})')
    ax6.plot(x_axis, ucb_total, color='black', linewidth=2,
             label='UCB = μ + κσ (acquisition)')

    # Ratio annotation: at the next point, what % is explore vs exploit?
    exploit_at_next = mu_next
    explore_at_next = kappa * std_next
    ucb_at_next     = exploit_at_next + explore_at_next
    exploit_pct     = 100 * exploit_at_next / ucb_at_next if ucb_at_next != 0 else 0
    explore_pct     = 100 * explore_at_next / ucb_at_next if ucb_at_next != 0 else 0

    ax6.axvline(next_x, color='green', linewidth=2, linestyle=':',
                label=f'Next point\n  UCB={ucb_at_next:.3f}\n'
                      f'  Exploit={exploit_pct:.0f}%\n'
                      f'  Explore={explore_pct:.0f}%')
    ax6.axvline(best_x, color='gold', linewidth=2, linestyle='-.',
                label='Current best')

    # Mark the acquisition maximum
    ucb_max_idx = np.argmax(ucb_total)
    ax6.scatter([x_axis[ucb_max_idx]], [ucb_total[ucb_max_idx]],
                marker='*', s=150, color='red', zorder=6, label='UCB max')

    ax6.set_xlabel(f'x{best_dim}')
    ax6.set_ylabel('Acquisition value')
    ax6.set_title(
        f'Panel 6 — UCB DECOMPOSITION\n'
        f'Next point: {exploit_pct:.0f}% exploit / {explore_pct:.0f}% explore'
    )
    ax6.legend(fontsize=6.5)

    # ── Row labels ────────────────────────────────────────────────────────────
    fig.text(0.01, 0.73, 'ROW 1\nModel\nAccuracy', fontsize=9, fontweight='bold',
             va='center', ha='left', color='#333333',
             bbox=dict(boxstyle='round', facecolor='#f0f0f0', alpha=0.6))
    fig.text(0.01, 0.27, 'ROW 2\nExplore\nvs\nExploit', fontsize=9, fontweight='bold',
             va='center', ha='left', color='#333333',
             bbox=dict(boxstyle='round', facecolor='#fff3e0', alpha=0.6))

    plt.savefig(f"{fn_label}_CNN_diagnostics.png", dpi=130, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {fn_label}_CNN_diagnostics.png")


# ══════════════════════════════════════════════════════════════════════════════
# PART 7: MAIN EXECUTION
# ══════════════════════════════════════════════════════════════════════════════

def run_all_functions(
    functions_to_run=None,
    use_cnn_amortisation=True,
    use_residual_correction=True,
    cnn_prescreen_k=5000,
    kappa=4.0,
    sv_weight=0.4,
    cnn_epochs=250,
    plot_diagnostics=True,
):
    """
    Run CNN-enhanced BBO for F1-F8 (or a subset).

    CNN enhancements are applied to all functions by default,
    but have the most impact on F4 and F5 due to high-curvature landscapes.
    """
    if functions_to_run is None:
        functions_to_run = list(range(1, 9))

    print("\n" + "=" * 70)
    print("BBO6: CNN-ENHANCED BBO  —  F1-F8")
    print(f"  Enhancement #4 (CNN Amortisation):   {use_cnn_amortisation}")
    print(f"  Enhancement #5 (Residual Correction): {use_residual_correction}")
    print(f"  CNN pre-screen k:                     {cnn_prescreen_k:,}")
    print(f"  CNN training epochs:                  {cnn_epochs}")
    print("=" * 70)

    all_results = {}

    for fn in functions_to_run:
        fn_label = f"F{fn}"
        print(f"\n\n{'#' * 70}")
        print(f"# FUNCTION {fn_label}")
        print(f"{'#' * 70}\n")

        try:
            X_obs = np.atleast_2d(np.load(f"f{fn}initial_inputs.npy"))
            y_obs = np.asarray(np.load(f"f{fn}initial_outputs.npy")).ravel()
            dim   = X_obs.shape[1]
            bounds = [(0.0, 1.0)] * dim
            print(f"Loaded: {len(y_obs)} observations, {dim}D")

            results = cnn_enhanced_bbo(
                X_obs, y_obs, bounds,
                kappa=kappa,
                sv_weight=sv_weight,
                n_random_samples=100_000,
                cnn_prescreen_k=cnn_prescreen_k,
                use_gradient_enhancement=True,
                use_cnn_amortisation=use_cnn_amortisation,
                use_residual_correction=use_residual_correction,
                fn_label=fn_label,
                cnn_epochs=cnn_epochs,
            )

            all_results[fn_label] = results

            # Save next point
            np.save(f"f{fn}_next_point_cnn_enhanced.npy", results['next_point'])
            print(f"\n✓ {fn_label} complete — next point saved")

            # Diagnostics plot
            if plot_diagnostics:
                plot_cnn_diagnostics(results, X_obs, y_obs, fn_label, bounds,
                                     kappa=kappa, sv_weight=sv_weight)

        except FileNotFoundError:
            print(f"\n✗ {fn_label}: data files not found — skipping")
        except Exception as e:
            import traceback
            print(f"\n✗ {fn_label}: failed — {e}")
            traceback.print_exc()

    # ── Summary table ─────────────────────────────────────────────────────────
    print(f"\n\n{'=' * 70}")
    print("SUMMARY")
    print(f"{'=' * 70}")
    header = f"{'Func':<6} {'Dim':<5} {'Kernel':<18} {'CNN#4':<7} {'CNN#5':<7} " \
             f"{'UCB':<12} {'Mean':<12} {'Std':<10}"
    print(header)
    print("-" * 70)
    for fn_label, res in sorted(all_results.items()):
        dim_str  = len(res['next_point'])
        cnn4_str = "YES" if res['grad_cnn']     is not None else "NO"
        cnn5_str = "YES" if res['residual_cnn'] is not None else "NO"
        print(f"{fn_label:<6} {dim_str:<5} {res['kernel']:<18} "
              f"{cnn4_str:<7} {cnn5_str:<7} "
              f"{res['ucb']:<12.4f} {res['mean']:<12.4f} {res['std']:<10.4f}")

    print(f"\n{'=' * 70}")
    print("NEXT QUERY POINTS")
    print(f"{'=' * 70}")
    for fn_label, res in sorted(all_results.items()):
        coords = ", ".join([f"{v:.6f}" for v in res['next_point']])
        print(f"\n{fn_label}: [{coords}]")

    return all_results


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    # Run all 8 functions with both CNN enhancements active.
    # To run only F4 and F5: set functions_to_run=[4, 5]
    results = run_all_functions(
        functions_to_run=list(range(1, 9)),
        use_cnn_amortisation=True,
        use_residual_correction=True,
        cnn_prescreen_k=5000,
        kappa=4.0,
        sv_weight=0.4,
        cnn_epochs=250,
        plot_diagnostics=True,
    )

    print("\n✓ ALL FUNCTIONS PROCESSED")
    print("Next points saved as: f1_next_point_cnn_enhanced.npy, etc.")
    print("Diagnostic plots saved as: F1_CNN_diagnostics.png, etc.")


PyTorch version: 2.9.1
Using device: cpu

BBO6: CNN-ENHANCED BBO  —  F1-F8
  Enhancement #4 (CNN Amortisation):   True
  Enhancement #5 (Residual Correction): True
  CNN pre-screen k:                     5,000
  CNN training epochs:                  250


######################################################################
# FUNCTION F1
######################################################################

Loaded: 15 observations, 2D
CNN-ENHANCED GP BBO  —  F1  (2D, 15 obs)

── STEP 1: KERNEL OPTIMISATION
  RBF_ARD: CV=-0.0000  kernel=1.03**2 * RBF(length_scale=[0.01, 20]) + WhiteKernel(noise_level=2.17e-09)
    ✓ New best
  Matern_2.5_ARD: CV=-0.0000  kernel=1.06**2 * Matern(length_scale=[0.0124, 20], nu=2.5) + WhiteKernel(noise_level=1.62e-09)
    ✓ New best
  Matern_1.5_ARD: CV=-0.0000  kernel=1.06**2 * Matern(length_scale=[0.0135, 20], nu=1.5) + WhiteKernel(noise_level=1.82e-10)
    ✓ New best

  Best kernel: Matern_1.5_ARD

── STEP 2: GRADIENT ENHANCEMENT
  Gradient magnitudes 